In [ ]:
import pandas as pd
import matplotlib
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import re

In [ ]:
# suppressing warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
data = pd.read_csv('train.csv/train.csv')
data

In [ ]:
data['OutcomeType'].value_counts().plot.bar()

In [ ]:
# Start cleaning the data 
# 1. AnimalId is probably not needed drop the column
# 2. Also it feels that OutcomeSubtype is kind of sub category of OutcomeType so I would want to drop it
data = data.drop(['AnimalID','OutcomeSubtype'], axis = 1)

In [ ]:
# check for nan values in all the columns and tackle them one by one
data.isna().sum()

In [ ]:
# Remove the Name column and refactor it. If name exists label it as '1' and if there is no name then label it '0'
data['Name'] = data[['Name']].fillna(value=0)

In [ ]:
data['newName'] = (data['Name'] != 0).astype('int64')

In [ ]:
data = data.drop(['Name'], axis = 1)

In [ ]:
# Check for null values now 
data.isna().sum()

In [ ]:
# As AnimalType contains only two types of class like dog and cat making use of get dummies (aka. one hot encoding) helps
data = pd.get_dummies(data,columns=['AnimalType'])

In [ ]:
# Now try to change theh color feature
data['Color'] = data['Color'].apply(lambda x: 1 if '/' in x else 0)

In [ ]:
# Try to change the breed column in a meaningful way
data['breed_type'] = data.Breed.str.extract(pat='({})'.format('|'.join(['Mix'])),flags=re.IGNORECASE,expand=False).str.lower().fillna('pure')

In [ ]:
data.iloc[2480,:]

In [ ]:
data = data.drop(['Breed'], axis = 1)

In [ ]:
# Change AgeuponOutcome to numeric as well 
def age_converter(row):
    str_age = row['AgeuponOutcome']
    [age, unit] = str_age.split(" ")
    unit = unit.lower()
    if 'day' in unit:
        if age == '0':
            return 1
        return int(age)
    if 'week' in unit:
        if age == '0':
            return 7
        return int(age) * 7
    if 'month' in unit:
        if age == '0':
            return 30
        return int(age) * 4 * 28
    if 'year' in unit:
        if age == '0':
            return 365
        return int(age) * 28 * 12
    

In [ ]:
data['AgeuponOutcome'] = data['AgeuponOutcome'].fillna('0 day')
data['age_in_number'] = data.apply(age_converter, axis=1)
data = data.drop('AgeuponOutcome', axis=1)

In [ ]:
# Drop DateTime as its not significant atleast for now
data = data.drop('DateTime', axis = 1)

In [ ]:
# One hot encode SexuponOutcome
data = pd.get_dummies(data,columns=['SexuponOutcome'])

In [ ]:
# One hot encode breed_type
data = pd.get_dummies(data, columns=['breed_type'])

In [ ]:
data

In [ ]:
# Split the preprocessed data into training and testing
X = data.iloc[:,1:13]
Y = data['OutcomeType']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size = 0.2, random_state = 42)

In [ ]:
# Using XGB classifier
model = XGBClassifier()
model.fit(X_train,y_train)

In [ ]:
# Predict
y_pred = model.predict(X_test)

In [ ]:
# Check for accuracy
accuracy =  accuracy_score(y_test,y_pred)
print("Accuracy: %.2f%%" % (accuracy * 100.0))